# Chapter 6.4 - Lazy Initialization

Most layers need to know their parameter shapes before they can store weights. Lazy initialization is a controlled exception: it lets a layer postpone part of its shape decision until real input reveals the missing dimension.

## How to use this notebook

Run the notebook from top to bottom. Every code block is designed to be cloud-runnable and self-contained inside this notebook. The drills are intentionally small: predict the shape or behavior first, run the cell, then read the assertion as the contract you must understand.

## You are done when you can

- explain why parameter shape depends on input shape
- identify uninitialized lazy parameters
- explain when lazy layer shapes become known
- distinguish convenience from a different model type
- use lazy layers in a clean-restartable notebook
- debug a lazy layer after it has locked onto an input size


In [ ]:
import math
from pathlib import Path
import tempfile

import torch
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_scalars(parameters):
    return sum(p.numel() for p in parameters)


## 6.4.0 The Problem This Notebook Solves

In a normal linear layer, the weight shape is known immediately:

```text
Linear(input_features, output_features)
weight shape: output_features by input_features
```

That means the layer designer must know the input feature count while writing the model. Sometimes this is easy. In an MLP over a fixed 10-feature table, the number is obvious. Sometimes it is annoying. After convolutions, pooling, flattening, or other shape-changing operations, the final feature count may take careful tracing.

Lazy initialization lets the model say:

```text
I know how many output features I want.
I will infer the input feature count when I see the first batch.
```

The theory is not that lazy layers learn differently. They do not. The theory is about delaying a software commitment. Before the first forward pass, the layer does not yet know enough to create an ordinary weight matrix. After the first forward pass, it has a real parameter shape and behaves like a regular layer.

The handoff from 6.3 is direct: initialization chooses parameter values, but parameter values require parameter shapes. Lazy modules postpone shape creation, then initialize once the missing shape is known.


## 6.4.1 Before the First Forward Pass, Some Shapes Are Unknown

`nn.LazyLinear(3)` knows that it should produce 3 output features. It does not know how many input features it will receive. Without that input feature count, it cannot create a complete weight matrix.

This is why inspecting a lazy weight too early is different from inspecting a normal parameter. The layer is not broken; it is incomplete by design.

Before running the cell, predict:

- Before the first forward pass, the weight shape should not be fully available.
- If the first input has shape `(2, 4)`, the layer sees 4 input features.
- The final weight shape should become `(3, 4)`.
- The output shape should become `(2, 3)`.


In [ ]:
lazy = nn.LazyLinear(3)

print("weight object type before forward:", type(lazy.weight).__name__)
try:
    print(lazy.weight.shape)
except RuntimeError as err:
    print("shape unavailable before forward:")
    print(str(err).splitlines()[0])

X = torch.randn(2, 4)
Y = lazy(X)

print("output shape:", shape(Y))
print("weight shape after forward:", shape(lazy.weight))

assert shape(Y) == (2, 3)
assert shape(lazy.weight) == (3, 4)


## 6.4.2 Lazy Layers Are Useful After Shape-Changing Layers

Flattening image-like tensors is a common place where lazy initialization is convenient.

An image batch shaped `(5, 1, 4, 4)` contains:

```text
5 examples
1 channel per example
4 rows
4 columns
```

After flattening, each example has `1 * 4 * 4 = 16` features. You can compute that by hand here, but in deeper convolutional networks the spatial size may change several times. A lazy linear layer can infer the flattened feature count from a sample forward pass.

The conceptual tradeoff is:

- lazy layer: easier to write when shape arithmetic is annoying
- explicit layer: clearer contract when teaching or debugging

The cell below uses lazy initialization only as a convenience. Once initialized, the layer owns ordinary parameters.


In [ ]:
net = nn.Sequential(
    nn.Flatten(),
    nn.LazyLinear(10),
)

X = torch.randn(5, 1, 4, 4)
Y = net(X)

print("flattened feature count:", net[1].weight.shape[1])
print("output shape:", shape(Y))

assert shape(Y) == (5, 10)
assert shape(net[1].weight) == (10, 16)


## 6.4.3 Explicit Dimensions Are Often Clearer in Teaching Code

Explicit dimensions make the architecture contract visible in the code. That is why teaching notebooks often prefer them even when lazy layers would work.

This comparison is important because lazy initialization can hide shape reasoning if used too early. You should still be able to explain why the flattened feature count is 16. Lazy layers should reduce boilerplate, not replace understanding.

The handoff to Chapter 7 is worth noticing: CNNs create many shape-changing steps. Lazy layers can make prototypes easier, but serious CNN debugging still requires tracing height, width, and channel count.


In [ ]:
explicit = nn.Sequential(
    nn.Flatten(),
    nn.Linear(16, 10),
)

X = torch.randn(5, 1, 4, 4)
Y = explicit(X)

print("explicit weight shape:", shape(explicit[1].weight))
print("lazy weight shape:", shape(net[1].weight))

assert shape(Y) == (5, 10)
assert shape(explicit[1].weight) == shape(net[1].weight)


## 6.4.4 Break It Deliberately: Change Feature Count After Initialization

Lazy does not mean permanently flexible.

After the first forward pass, the layer has committed to a weight matrix. If the first input had 4 features, the weight expects 4 features from then on. Passing 5 features later is not a new lazy inference event; it is a shape mismatch.

This is the central warning:

```text
lazy initialization delays the first shape decision
it does not remove shape contracts
```


In [ ]:
lazy = nn.LazyLinear(3)
lazy(torch.randn(2, 4))

try:
    lazy(torch.randn(2, 5))
except RuntimeError as err:
    print(type(err).__name__)
    print(str(err).splitlines()[0])
else:
    raise AssertionError("The initialized lazy layer should reject a new feature count.")


## 6.4 Checkpoint

Answer these before moving on. You do not need a separate notes file for chapters; short answers in markdown cells or in your own study notes are enough.

1. Why can a linear layer not create its full weight matrix without knowing input feature count?
2. Which dimension is unknown in `nn.LazyLinear(3)` before the first forward pass?
3. What input shape made the lazy weight become `(3, 4)`?
4. Why is lazy initialization a software convenience rather than a new learning rule?
5. Why can a lazy layer still fail later with a shape mismatch?
6. When would you prefer explicit dimensions over lazy initialization?
